In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# ── Configuration ────────────────────────────────────────────────────────────
SA_NAME = "stmaritimeks"
BRONZE_VESSELS_PATH = f"abfss://bronze@{SA_NAME}.dfs.core.windows.net/gfw/vessels/load_type=*/ingest_date=*/*.jsonl"
SILVER_DELTA_PATH = f"abfss://silver@{SA_NAME}.dfs.core.windows.net/stg_vessels"
SILVER_TABLE_NAME = "maritime_dw.silver.stg_vessels"  # Fallback handled below if catalog differs


def clean_str(col_expr):
    """Trim whitespace, collapse internal multiple spaces, and strip trailing garbage symbols."""
    cleaned = F.regexp_replace(F.trim(col_expr), r"\s+", " ")
    cleaned = F.regexp_replace(cleaned, r"[>&*#]+$", "")
    cleaned = F.trim(cleaned)
    return F.when(cleaned.isNull() | (cleaned == ""), F.lit(None)).otherwise(cleaned)


def run_stg_vessels():
    raw_df = spark.read.json(BRONZE_VESSELS_PATH)

    # 1. Explode selfReportedInfo to map every event vessel_id to its parent record
    sri_exploded = (
        raw_df
        .withColumn("_source_file", F.input_file_name())
        .withColumn("sri", F.explode_outer("selfReportedInfo"))
        .filter(F.col("sri.id").isNotNull())
    )

    # 2. Extract the most recent combinedSourcesInfo entry (highest yearTo) per vessel_id
    csi_exploded = (
        sri_exploded
        .select(
            F.col("sri.id").alias("vessel_id"),
            F.col("sri"),
            F.col("_source_file"),
            F.explode_outer("combinedSourcesInfo").alias("csi"),
        )
    )

    latest_csi_window = Window.partitionBy("vessel_id").orderBy(
        F.col("csi.yearTo").desc_nulls_last(),
        F.col("csi.yearFrom").desc_nulls_last(),
        F.col("sri.transmissionDateTo").desc_nulls_last(),
    )

    deduped_df = (
        csi_exploded
        .withColumn("_rn", F.row_number().over(latest_csi_window))
        .filter(F.col("_rn") == 1)
        .drop("_rn")
    )

    # 3. Helper to extract the most recent value/name from array-of-struct fields in combinedSourcesInfo
    def latest_metric_value(arr_col, field_name="value"):
        sorted_arr = F.array_sort(
            F.transform(
                arr_col,
                lambda x: F.struct(
                    F.coalesce(x["yearTo"], F.lit(0)).alias("yr"),
                    x[field_name].alias("val"),
                ),
            )
        )
        return F.element_at(sorted_arr, -1)["val"]

    # 4. Build standardized Silver staging columns
    stg_df = deduped_df.select(
        F.trim(F.col("vessel_id")).alias("vessel_id"),
        F.coalesce(
            clean_str(F.col("sri.shipname")),
            clean_str(F.col("sri.nShipname")),
            F.lit("Unknown"),
        ).alias("vessel_name"),
        F.coalesce(clean_str(F.col("sri.ssvid")), F.lit("Unknown")).alias("mmsi"),
        F.coalesce(clean_str(F.col("sri.imo")), F.lit("Unknown")).alias("imo"),
        F.coalesce(clean_str(F.col("sri.callsign")), F.lit("Unknown")).alias("callsign"),
        F.upper(F.coalesce(clean_str(F.col("sri.flag")), F.lit("UNK"))).alias("flag_iso3"),
        F.coalesce(
            clean_str(latest_metric_value(F.col("csi.shiptypes"), "name")),
            F.lit("UNKNOWN"),
        ).alias("vessel_type"),
        F.coalesce(
            clean_str(latest_metric_value(F.col("csi.geartypes"), "name")),
            F.lit("UNKNOWN"),
        ).alias("gear_type"),
        latest_metric_value(F.col("csi.bestLengthM"), "value").cast("decimal(10,2)").alias("length_m"),
        latest_metric_value(F.col("csi.bestTonnageGt"), "value").cast("decimal(12,2)").alias("tonnage_gt"),
        latest_metric_value(F.col("csi.bestEnginePowerKw"), "value").cast("decimal(12,2)").alias("engine_power_kw"),
        latest_metric_value(F.col("csi.maxSpeedKn"), "value").cast("decimal(8,2)").alias("max_speed_knots"),
        F.col("csi.yearFrom").cast("int").alias("registry_year_from"),
        F.col("csi.yearTo").cast("int").alias("registry_year_to"),
        F.to_timestamp(F.col("sri.transmissionDateFrom")).alias("first_transmission_utc"),
        F.to_timestamp(F.col("sri.transmissionDateTo")).alias("last_transmission_utc"),
        F.current_timestamp().alias("silver_loaded_at_utc"),
        F.col("_source_file").alias("bronze_source_file"),
    ).dropDuplicates(["vessel_id"])

    # 5. Write to Silver Delta Lake & register table
    (
        stg_df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .save(SILVER_DELTA_PATH)
    )

    try:
        spark.sql("CREATE SCHEMA IF NOT EXISTS maritime_dw.silver")
        spark.sql(f"""
            CREATE TABLE IF NOT EXISTS {SILVER_TABLE_NAME}
            USING DELTA
            LOCATION '{SILVER_DELTA_PATH}'
        """)
        print(f"Registered Unity Catalog table: {SILVER_TABLE_NAME}")
    except Exception as e:
        print(f"[INFO] Skipped Unity Catalog table registration (Delta written to ADLS): {e}")

    total_rows = spark.read.format("delta").load(SILVER_DELTA_PATH).count()
    print(f"Success! Wrote {total_rows} clean vessel records to {SILVER_DELTA_PATH}")


if __name__ == "__main__":
    run_stg_vessels()